In [38]:
"""
medir_separabilidad_llm.py
============================

Mide directamente si los EMBEDDINGS de cada modelo separan los sentidos que
el juez LLM identificó -- sin pasar por HDBSCAN ni UMAP.

Motivación (reemplaza/complementa comparar_llm_clustering.py): comparar
ARI/NMI contra los clusters de HDBSCAN mezclaba dos preguntas distintas:
  (1) ¿el embedding contiene la información para distinguir sentidos?
  (2) ¿HDBSCAN sobre una proyección UMAP logra RECUPERAR esa estructura con
      min_cluster_size, el umbral de ruido del 50%, etc.?
El ruido propio de (2) -- ya documentado extensamente en este protocolo
(no-determinismo de UMAP, sesgo contra sentidos minoritarios por el piso de
min_cluster_size) -- se mezclaba con la señal de (1) en las métricas
anteriores. Aquí se evalúa (1) directamente: dado el embedding CRUDO (sin
reducir dimensión, sin clustering) y las etiquetas de sentido "reales" del
juez LLM (ya sin ruido -- reusa el filtro de comparar_llm_clustering.py),
¿tan separables son esas clases en el espacio de embeddings?

Dos medidas, ambas SUPERVISADAS (usan las etiquetas del LLM directamente,
nunca una partición propia del modelo):
  - silhouette score sobre el embedding crudo, con las etiquetas del LLM como
    verdad -- qué tan compactos/separados son los grupos geométricamente.
  - accuracy de un clasificador k-NN con validación leave-one-out, entrenado a
    predecir el sentido del LLM a partir del embedding -- más intuitivo que
    silhouette ("¿se puede adivinar el sentido viendo solo el embedding?").

Ninguna de las dos tiene una escala absoluta interpretable por sí sola (un
silhouette de 0.1 puede ser mucho o poco dependiendo del tamaño y balance de
las clases), así que ambas se comparan contra una distribución NULA por
permutación: se baraja la etiqueta del LLM entre los mismos tweets (manteniendo
el tamaño de cada clase) N_PERMUTACIONES veces, y se reporta el percentil del
valor observado dentro de esa distribución nula. Un percentil >=0.95 en una
palabra-modelo es evidencia de que el embedding separa esos sentidos mejor de
lo esperado por azar puro, dado el tamaño de las clases -- sin necesidad de
justificar qué cuenta como "silhouette alto" en abstracto.

Requiere haber corrido antes: llm_juez_local.py (juicios_llm/<palabra>.json).

Uso:
    df = evaluar_lote(
        palabras=["entrada", "verde", "elefante", ...],
        construir_dicts_fn=construir_dicts_para_palabras,   # de clusterizar_sentidos.py
        obtener_vectores_fn=mi_funcion_de_vectores,         # ver abajo -- adaptar a tu formato real
    )
"""

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from itertools import combinations

import os
import unicodedata
from collections import defaultdict
import json
import glob

In [14]:
CARPETA_JUICIOS = "juicios_llm_qwen_model_"
CARPETA_EMBEDDINGS = "embeddings"
MIN_TWEETS_POR_SENTIDO_LLM = 5
 
ETIQUETA_RUIDO_LLM = "ruido_llm"

def _asignaciones_con_ruido_llm(juicio, min_tweets=MIN_TWEETS_POR_SENTIDO_LLM):
    """Devuelve (asignaciones_llm, sentidos_ruido, n_sentidos_5plus).
 
    asignaciones_llm: dict id_tweet -> sentido, con los sentidos que tienen
    menos de `min_tweets` tweets asignados (contando TODA la muestra que vio
    el juez, no solo la que se cruza con el modelo) re-etiquetados como
    ETIQUETA_RUIDO_LLM.
    sentidos_ruido: el conjunto de ids de sentido tratados como ruido.
    n_sentidos_5plus: cuántos sentidos declarados por el LLM tienen >=
    min_tweets tweets -- el conteo "real" de sentidos una vez descontado el
    ruido, más comparable con el criterio que ya se usa para decidir si un
    modelo de clustering "presenta múltiples clusters" (sección 8/9).
    """
    conteo = {}
    for a in juicio["asignaciones"]:
        conteo[a["sentido"]] = conteo.get(a["sentido"], 0) + 1
 
    sentidos_ruido = {
        sid for sid, n in conteo.items()
        if sid != "no_claro" and n < min_tweets
    }
    n_sentidos_5plus = sum(
        1 for sid in (s["id"] for s in juicio.get("sentidos", []))
        if conteo.get(sid, 0) >= min_tweets
    )
 
    asignaciones_llm = {
        a["id_tweet"]: (ETIQUETA_RUIDO_LLM if a["sentido"] in sentidos_ruido else a["sentido"])
        for a in juicio["asignaciones"]
    }
    return asignaciones_llm, sentidos_ruido, n_sentidos_5plus


def cargar_juicio_llm(palabra, carpeta=CARPETA_JUICIOS):
    ruta = os.path.join(carpeta, f"{palabra}.json")
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

        
def cargar_embeddings(nombre_modelo, carpeta=CARPETA_EMBEDDINGS):
    ruta = os.path.join(carpeta, f"{nombre_modelo}_embeddings.npz")
    data = np.load(ruta, allow_pickle=True)
    emb = data["embeddings"]
    words = data["words"]
    id_tweets = data["id_tweets"]
    return emb, words, id_tweets

In [40]:
N_PERMUTACIONES = 1000  # con vecinos/distancias precalculados esto es barato; no bajar de ~200
K_VECINOS = 5
SEMILLA = 42
ALFA = 0.05


def obtener_vectores_fn(palabra, embeddings_dict, tweets_dict):
    """PLACEHOLDER -- adapta esta función al formato real de embeddings_dict
    que usa construir_dicts_para_palabras() en clusterizar_sentidos.py. Debe
    devolver un dict {id_tweet: vector_embedding (np.array)} para `palabra`.
    No se pudo escribir la versión final sin ver esa estructura exacta -- pero
    debería ser una o dos líneas, ya que inspeccionar_ocurrencias_fn ya hace
    este mismo lookup internamente para construir su tabla id_tweet/cluster_id."""
    embs = np.array(embeddings_dict[palabra])
    tids = tweets_dict[palabra]
    res = {}
    for i in range(len(embs)):
        res[tids[i]] = embs[i]
    return res
    
def _vecinos_loo(D, k):
    """Índices de los k vecinos más cercanos de cada punto, excluyéndose a sí
    mismo (leave-one-out). Los vecinos dependen solo de las distancias, NO de
    las etiquetas -- por eso se calculan una sola vez y se reutilizan en todas
    las permutaciones (antes se reentrenaba un k-NN por punto y por
    permutación, lo que hacía impráctico usar cientos de permutaciones)."""
    D = D.copy()
    np.fill_diagonal(D, np.inf)
    k_efectivo = min(k, D.shape[0] - 1)
    return np.argsort(D, axis=1, kind="stable")[:, :k_efectivo]

def _accuracy_loo_knn(vecinos, y_codificado, n_clases):
    """Accuracy de k-NN leave-one-out dado los vecinos precalculados. Voto
    mayoritario uniforme; en empate gana la clase de menor índice (mismo
    criterio que KNeighborsClassifier de sklearn)."""
    etiquetas_vecinos = y_codificado[vecinos]  # (n, k)
    votos = np.zeros((len(y_codificado), n_clases), dtype=int)
    for c in range(n_clases):
        votos[:, c] = (etiquetas_vecinos == c).sum(axis=1)
    pred = votos.argmax(axis=1)
    return float((pred == y_codificado).mean())


def evaluar_palabra_modelo(palabra, nombre_modelo, embeddings_dict, tweets_dict,
                             obtener_vectores_fn, carpeta_juicios=CARPETA_JUICIOS,
                             min_tweets_sentido_llm=MIN_TWEETS_POR_SENTIDO_LLM,
                             n_permutaciones=N_PERMUTACIONES, k_vecinos=K_VECINOS,
                             semilla=SEMILLA):
    """Para UNA palabra y UN modelo: arma (X, y) con los embeddings crudos y
    las etiquetas de sentido del LLM (ya sin "no_claro" ni sentidos-ruido de
    <5 tweets), calcula silhouette y accuracy k-NN observados, y los compara
    contra una distribución nula por permutación de etiquetas."""
    juicio = cargar_juicio_llm(palabra, carpeta_juicios)
    asignaciones_llm, _, n_sentidos_5plus = _asignaciones_con_ruido_llm(
        juicio, min_tweets=min_tweets_sentido_llm
    )
 
    vectores_por_tweet = obtener_vectores_fn(palabra, embeddings_dict, tweets_dict)
 
    X_lista, y_lista = [], []
    for id_tweet, sentido in asignaciones_llm.items():
        if sentido in ("no_claro", ETIQUETA_RUIDO_LLM):
            continue
        vector = vectores_por_tweet.get(id_tweet)
        if vector is None:
            continue
        X_lista.append(vector)
        y_lista.append(sentido)
 
    base = {
        "palabra": palabra, "modelo": nombre_modelo,
        "n_sentidos_llm_5plus": n_sentidos_5plus, "n_comparado": len(y_lista),
    }
 
    vacio = {**base, "silhouette": np.nan, "accuracy_knn": np.nan,
             "accuracy_base": np.nan, "mejora_sobre_base": np.nan,
             "p_silhouette": np.nan, "p_accuracy": np.nan}
 
    if len(y_lista) < 10:
        return vacio
 
    y = np.array(y_lista)
    clases, conteos = np.unique(y, return_counts=True)
    if len(clases) < 2 or conteos.min() < 2:
        # una sola clase real, o alguna clase con un solo ejemplo -- ni
        # silhouette ni k-NN leave-one-out están bien definidos
        return vacio
 
    X = np.array(X_lista)
    y_codificado = np.searchsorted(clases, y)  # etiquetas como enteros 0..n_clases-1
 
    # Distancias y vecinos se calculan UNA vez: no dependen de las etiquetas.
    D = pairwise_distances(X, metric="euclidean")
    vecinos = _vecinos_loo(D, k_vecinos)
 
    sil_obs = silhouette_score(D, y_codificado, metric="precomputed")
    acc_obs = _accuracy_loo_knn(vecinos, y_codificado, len(clases))
    acc_base = conteos.max() / len(y)  # predecir siempre la clase mayoritaria
 
    rng = np.random.default_rng(semilla)
    sil_null = np.empty(n_permutaciones)
    acc_null = np.empty(n_permutaciones)
    for i in range(n_permutaciones):
        y_perm = rng.permutation(y_codificado)
        sil_null[i] = silhouette_score(D, y_perm, metric="precomputed")
        acc_null[i] = _accuracy_loo_knn(vecinos, y_perm, len(clases))
 
    # p-valor por permutación con la corrección estándar (+1 en numerador y
    # denominador): nunca da p=0, y con B permutaciones el mínimo alcanzable
    # es 1/(B+1). Con B=10 ese mínimo es ~0.09 -- por eso B debe ser >=~200
    # para poder hablar de p<0.05 con resolución razonable.
    p_sil = float(((sil_null >= sil_obs).sum() + 1) / (n_permutaciones + 1))
    p_acc = float(((acc_null >= acc_obs).sum() + 1) / (n_permutaciones + 1))
 
    return {
        **base,
        "silhouette": sil_obs, "accuracy_knn": acc_obs, "accuracy_base": acc_base,
        "mejora_sobre_base": acc_obs - acc_base,
        "p_silhouette": p_sil, "p_accuracy": p_acc,
    }
 
 
def evaluar_lote(palabras, construir_dicts_fn, obtener_vectores_fn,
                   modelos=("mex_state", "mex_large", "beto", "bertin", "robertuito"),
                   carpeta_juicios=CARPETA_JUICIOS,
                   min_tweets_sentido_llm=MIN_TWEETS_POR_SENTIDO_LLM,
                   n_permutaciones=N_PERMUTACIONES, k_vecinos=K_VECINOS):
    """`construir_dicts_fn` es construir_dicts_para_palabras() de
    clusterizar_sentidos.py. `obtener_vectores_fn` debe devolver, para una
    palabra, {id_tweet: vector} -- adaptar al formato real de embeddings_dict
    (ver obtener_vectores_fn_ejemplo arriba)."""
    resultados = []
    for nombre_modelo in modelos:
        emb_dict, tw_dict = construir_dicts_fn(nombre_modelo, set(palabras))
        for palabra in palabras:
            if palabra not in emb_dict:
                print(f"  [{nombre_modelo}] '{palabra}' no está en los embeddings -- se salta")
                continue
            r = evaluar_palabra_modelo(
                palabra, nombre_modelo, emb_dict, tw_dict, obtener_vectores_fn,
                carpeta_juicios, min_tweets_sentido_llm, n_permutaciones, k_vecinos,
            )
            resultados.append(r)
 
    df = pd.DataFrame(resultados)
    df.to_csv("separabilidad_llm_vs_embeddings.csv", index=False)
 
    print("=== Separabilidad de los sentidos del LLM en el espacio de embeddings crudo, por modelo ===")
    print(df.groupby("modelo")[["silhouette", "accuracy_knn", "accuracy_base", "mejora_sobre_base"]]
          .describe([0.5]).to_string())
 
    print(f"\n=== % de palabras separables mejor que el azar (p < {ALFA}, {n_permutaciones} permutaciones) ===")
    for modelo, grupo in df.groupby("modelo"):
        valido = grupo.dropna(subset=["p_silhouette"])
        if len(valido):
            tasa_sil = (valido["p_silhouette"] < ALFA).mean()
            tasa_acc = (valido["p_accuracy"] < ALFA).mean()
            print(f"  {modelo}: silhouette {tasa_sil:.1%}, accuracy k-NN {tasa_acc:.1%} (n={len(valido)})")
 
    comparar_modelos_pareado(df)
    return df


def comparar_modelos_pareado(df, metricas=("mejora_sobre_base", "silhouette")):
    """Compara modelos sobre las MISMAS palabras (diseño pareado): Friedman
    global entre los 5 modelos y Wilcoxon por pares con corrección de
    Bonferroni (10 comparaciones, igual que en la sección 6 del protocolo).
    Comparar medias sueltas entre modelos ignora que la dificultad varía
    mucho de palabra a palabra; el pareo controla eso."""
    for metrica in metricas:
        tabla = df.pivot_table(index="palabra", columns="modelo", values=metrica).dropna()
        modelos = list(tabla.columns)
        if len(tabla) < 5 or len(modelos) < 3:
            print(f"\n[{metrica}] muy pocas palabras comunes a todos los modelos ({len(tabla)}) -- se omite")
            continue
 
        print(f"\n=== Comparación pareada entre modelos: {metrica} (n={len(tabla)} palabras comunes) ===")
        print("Mediana por modelo: " + ", ".join(f"{m}={tabla[m].median():.3f}" for m in modelos))
        stat, p = friedmanchisquare(*[tabla[m] for m in modelos])
        print(f"Friedman: chi2={stat:.2f}, p={p:.4g}")
 
        pares = list(combinations(modelos, 2))
        alfa_bonf = ALFA / len(pares)
        for a, b in pares:
            dif = tabla[a] - tabla[b]
            if (dif == 0).all():
                print(f"  {a} vs {b}: sin diferencias")
                continue
            _, p_par = wilcoxon(tabla[a], tabla[b])
            marca = " *" if p_par < alfa_bonf else ""
            print(f"  {a} vs {b}: mediana dif={dif.median():+.3f}, p={p_par:.4g}{marca}")
        print(f"  (* = significativo con Bonferroni, alfa={alfa_bonf:.4f})")


def construir_dicts(nombre_modelo, palabras):
    emb, w, id_tweets= cargar_embeddings(nombre_modelo)
    emb /= np.linalg.norm(emb, axis=1).reshape(-1, 1)

    # normaliza a NFC -- "ñ" puede venir como un solo carácter o como "n" + tilde
    # combinada; se ven idénticas pero no son el mismo string en Python
    palabras_arr = np.array([unicodedata.normalize("NFC", str(d)) for d in w])
    tweets_arr = np.array(id_tweets)

    emb_dict, tw_dict = defaultdict(list), defaultdict(list)
    for palabra in palabras:
        palabra_nfc = unicodedata.normalize("NFC", palabra)
        idx = np.where(palabras_arr == palabra_nfc)[0]
        if len(idx) == 0:
            print(f"  [{nombre_modelo}] '{palabra}' no está en el .npz guardado — sáltala o revisa el nombre")
            continue
        emb_dict[palabra] = list(emb[idx])
        tw_dict[palabra] = list(tweets_arr[idx])
    #embeddings_calib_por_modelo[nombre_modelo] = emb_dict
    #tweets_calib_por_modelo[nombre_modelo] = tw_dict
    return emb_dict, tw_dict

In [31]:
model = "qwen_model/"
files = glob.glob("./juicios_llm_" + model.replace("/", "_") + "/*")
palabras = [f.split('\\')[-1].split('.')[0] for f in files]

In [39]:
#palabras = ['ando']
construir_dicts_fn = construir_dicts
obtener_vectores = obtener_vectores_fn
df = evaluar_lote(palabras, construir_dicts_fn, obtener_vectores_fn,
                   modelos=("mex_state", "mex_large", "beto", "bertin", "robertuito"),
                   carpeta_juicios=CARPETA_JUICIOS,
                   min_tweets_sentido_llm=MIN_TWEETS_POR_SENTIDO_LLM,
                   n_permutaciones=N_PERMUTACIONES, k_vecinos=K_VECINOS)

=== Separabilidad de los sentidos del LLM en el espacio de embeddings crudo, por modelo ===
           silhouette                                                                       accuracy_knn                                                                  accuracy_base                                                                       mejora_sobre_base                                                                      
                count      mean       std       min       25%       50%       75%       max        count      mean       std       min       25%       50%       75%  max         count      mean       std       min       25%       50%       75%       max             count      mean       std       min       25%       50%       75%       max
modelo                                                                                                                                                                                                                          

,palabra,modelo,n_sentidos_llm_5plus,n_comparado,silhouette,accuracy_knn,accuracy_base,mejora_sobre_base,p_silhouette,p_accuracy
0,ando,mex_state,2,35,0.032254,0.742857,0.742857,0.000000,0.042957,0.206793
1,banco,mex_state,2,32,0.478132,0.968750,0.843750,0.125000,0.000999,0.000999
2,busca,mex_state,2,40,0.056160,0.800000,0.600000,0.200000,0.004995,0.000999
3,california,mex_state,2,39,0.297304,0.974359,0.871795,0.102564,0.000999,0.000999
4,carita,mex_state,1,33,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
295,tuve,robertuito,2,40,0.095505,1.000000,0.750000,0.250000,0.000999,0.000999
296,urge,robertuito,2,40,0.143787,0.950000,0.575000,0.375000,0.000999,0.000999
297,vacuna,robertuito,1,38,NaN,NaN,NaN,NaN,NaN,NaN
298,verde,robertuito,4,31,0.040743,0.548387,0.354839,0.193548,0.000999,0.000999


In [41]:
import pandas as pd
df = pd.read_csv("separabilidad_llm_vs_embeddings.csv")
genuina = {"entrada","verde","fuerza","temporada","lucha","esperar","leer","acabo","elefante","rico"}
tema = {"pagar","jugar","condiciones","viaje","busca","vida","confianza","chido","carita","homenaje","california","saltillo","juan"}
df["categoria"] = df["palabra"].map(lambda p: "genuina" if p in genuina else "tema" if p in tema else None)
sub = df.dropna(subset=["categoria", "p_accuracy"])
print(sub.groupby(["categoria","modelo"]).agg(n=("palabra","size"),
      pct_separable=("p_accuracy", lambda s: (s < 0.05).mean()),
      mejora_mediana=("mejora_sobre_base","median")).to_string())

                      n  pct_separable  mejora_mediana
categoria modelo                                      
genuina   bertin      4           0.75        0.157258
          beto        4           1.00        0.352431
          mex_large   4           1.00        0.247372
          mex_state   4           1.00        0.183081
          robertuito  4           0.75        0.207885
tema      bertin      5           0.60        0.054054
          beto        5           0.80        0.060606
          mex_large   5           0.80        0.121212
          mex_state   5           1.00        0.102564
          robertuito  5           0.40        0.026316
